# Rate Capability Processor

NDA -> chart workbook plus Stage 1 fast-path pkl.

**5 features**:
- `rate_cap_ratio_3C_vs_0.5C` (INPUT)
- `rate_cap_voltage_drop_3C_vs_0.5C_V` (INPUT)
- `rate_cap_voltage_drop_5C_vs_0.5C_V` (INPUT)
- `rate_cap_trapped_Li_fraction_after_5C` (INPUT) - **= 1 - Q_chg / Q_disch** at recovery cycle
- `rate_cap_ratio_5C_vs_0.5C` (OUTPUT)


# 1. NDA folder

In [ ]:
import os, re
import pandas as pd
import numpy as np
import traitlets, matplotlib
from ipywidgets import widgets
from IPython.display import display, clear_output
import warnings; warnings.filterwarnings("ignore")
import NewareNDA

def _pick_dir():
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw()
        try: root.call("wm","attributes",".","-topmost",True)
        except Exception: pass
        d = filedialog.askdirectory()
        try: root.update(); root.destroy()
        except Exception: pass
        return d or None
    except Exception as e:
        print(f"  (Tk picker failed: {e}; type the path manually instead)"); return None

def _pick_file():
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk(); root.withdraw()
        try: root.call("wm","attributes",".","-topmost",True)
        except Exception: pass
        p = filedialog.askopenfilename()
        try: root.update(); root.destroy()
        except Exception: pass
        return p or None
    except Exception as e:
        print(f"  (Tk picker failed: {e}; paste the file path in the text box)"); return None

class _DirBtn(widgets.Button):
    def __init__(self, label, target):
        super().__init__()
        self.description = label; self.icon = "folder-open-o"; self.style.button_color = "orange"
        self._t = target; self.on_click(self._click)
    def _click(self, b):
        with _out:
            clear_output(); d = _pick_dir()
            if d:
                self._t.value = d; b.style.button_color = "lightgreen"; b.description = "Selected"; print(d)

class _FileBtn(widgets.Button):
    def __init__(self, label, target):
        super().__init__()
        self.description = label; self.icon = "file-excel-o"; self.style.button_color = "orange"
        self._t = target; self.on_click(self._click)
    def _click(self, b):
        with _out:
            clear_output(); p = _pick_file()
            if p:
                self._t.value = p; b.style.button_color = "lightgreen"; b.description = "Selected"; print(p)

_out = widgets.Output()
nda_text = widgets.Text(description="NDA folder:", layout=widgets.Layout(width="900px"),
                        placeholder="Click button OR paste path here")
display(widgets.VBox([_DirBtn("Select NDA folder", nda_text), nda_text, _out]))


# 2. Metadata

In [ ]:
_out2 = widgets.Output()
meta_text = widgets.Text(description="metadata xlsx:", layout=widgets.Layout(width="900px"),
                         placeholder="Click or paste path")
display(widgets.VBox([_FileBtn("Select metadata xlsx", meta_text), meta_text, _out2]))


# 3. Cycle map (with C-rate tags)

In [ ]:
_out3 = widgets.Output()
cmap_text = widgets.Text(description="cycle map xlsx:", layout=widgets.Layout(width="900px"),
                         placeholder="Click or paste path")
display(widgets.VBox([_FileBtn("Select rate-cap cycle map xlsx", cmap_text), cmap_text, _out3]))


# 4. Output folder + filename

In [ ]:
_out4 = widgets.Output()
out_dir_text = widgets.Text(description="output folder:", layout=widgets.Layout(width="900px"),
                            placeholder="Click or paste path")
fname_text   = widgets.Text(value="rate_cap_output.xlsx", description="filename:",
                            layout=widgets.Layout(width="500px"))
display(widgets.VBox([_DirBtn("Select output folder", out_dir_text), out_dir_text, fname_text, _out4]))


# 5. Process

In [ ]:
RATE_TAG_REGEX = re.compile(r"(?:^|[\s_])(\d*\.?\d+)\s*C(?:[\s_]|$)", re.IGNORECASE)
RECOVERY_HINTS = ("recovery","recover","after_5c","post_5c")

def parse_rate_tag(s):
    if not isinstance(s, str): return None, False
    sl = s.lower(); is_recov = any(h in sl for h in RECOVERY_HINTS)
    m = RATE_TAG_REGEX.search(sl)
    return (None, is_recov) if not m else (f"{float(m.group(1)):g}C", is_recov)

def load_rate_cycle_map(path):
    cm = pd.read_excel(path, engine="openpyxl"); cm.columns = [str(c).strip() for c in cm.columns]
    nw_col = next((c for c in cm.columns if c.lower() in ("nw cycles","nw cycle","cycle","neware cycle","nw_cycle")), cm.columns[0])
    state_col = next((c for c in cm.columns if c.lower() in ("state","c-rate","crate","rate","note","tag")), None)
    if state_col is None:
        for c in cm.columns[::-1]:
            if cm[c].dtype == object: state_col = c; break
    cm["_nw_cycle"] = pd.to_numeric(cm[nw_col], errors="coerce")
    cm["_tag"] = cm[state_col].astype(str)
    cm[["_rate","_is_recov"]] = cm["_tag"].apply(lambda s: pd.Series(parse_rate_tag(s)))
    cm = cm.dropna(subset=["_nw_cycle","_rate"]).reset_index(drop=True)
    return cm[["_nw_cycle","_rate","_is_recov","_tag"]]

def find_last_cycle(cm, target_rate, prefer_recovery=False):
    sub = cm[cm["_rate"] == target_rate]
    if prefer_recovery and sub["_is_recov"].any(): sub = sub[sub["_is_recov"]]
    elif not prefer_recovery and sub["_is_recov"].any() and (~sub["_is_recov"]).any(): sub = sub[~sub["_is_recov"]]
    return None if sub.empty else int(sub["_nw_cycle"].max())

def find_recovery_cycle(cm, after_rate="5C"):
    fives = cm[cm["_rate"] == after_rate]
    if fives.empty: return None
    last = fives["_nw_cycle"].max()
    half = cm[(cm["_rate"] == "0.5C") & (cm["_nw_cycle"] > last)]
    return None if half.empty else int(half["_nw_cycle"].min())

def per_cycle_summary(rd):
    rd = rd.copy(); rd["Cycle"] = pd.to_numeric(rd["Cycle"], errors="coerce")
    dchg = rd[rd["Status"].isin(["CC_DChg","CCCV_DChg"])].groupby("Cycle").agg(
        disch_cap=("Discharge_Capacity(mAh)","max"), disch_energy=("Discharge_Energy(mWh)","max"))
    chg = rd[rd["Status"].isin(["CC_Chg","CCCV_Chg"])].groupby("Cycle").agg(
        chg_cap=("Charge_Capacity(mAh)","max"), chg_energy=("Charge_Energy(mWh)","max"))
    out = chg.join(dchg, how="outer").reset_index()
    out["V_avg_discharge"] = np.where(out["disch_cap"]>0, out["disch_energy"]/out["disch_cap"], np.nan)
    out["V_avg_charge"]    = np.where(out["chg_cap"]>0,   out["chg_energy"]/out["chg_cap"],    np.nan)
    out["CE_pct"] = np.where(out["chg_cap"]>0, out["disch_cap"]/out["chg_cap"]*100.0, np.nan)
    return out

# Read paths
nda_dir = nda_text.value.strip()
metadata_path = meta_text.value.strip()
cycle_map_path = cmap_text.value.strip()
out_dir = out_dir_text.value.strip()
fname = fname_text.value.strip() or "rate_cap_output.xlsx"
if not fname.lower().endswith(".xlsx"): fname += ".xlsx"
out_xlsx = os.path.join(out_dir, fname)
out_pkl  = os.path.splitext(out_xlsx)[0] + ".stage1ready.pkl"

assert os.path.isdir(nda_dir),       f"NDA folder not found: {nda_dir!r}"
assert os.path.isfile(metadata_path), f"metadata not found: {metadata_path!r}"
assert os.path.isfile(cycle_map_path),f"cycle map not found: {cycle_map_path!r}"
assert os.path.isdir(out_dir),       f"output folder not found: {out_dir!r}"

metadata = pd.read_excel(metadata_path); metadata.columns = [str(c).strip() for c in metadata.columns]
bc_col = next((c for c in metadata.columns if c.lower()=="barcode"), metadata.columns[0])
group_col = next((c for c in metadata.columns if c.lower() in ("group","electrolyte","el id")), metadata.columns[1])

PALETTE = ["#000000","#008463","#faaa00","#006384","#8A2DFB","#640a32","#ffd700","#B2B2B2",
           "#FF2525","#520DFF","#00FF00","#E49EDD","#00FFFF","#996633","#666699","#C4009A",
           "#000066","#d4e157","#83E28E","#ff6f00","#64ffda","#f50057","#e3f2fd","#D6AD84",
           "#920095","#808000","#FFFF00","#009900","#3399FF","#A52A2A","#5F9EA0","#FF69B4",
           "#BC8F8F","#DFCAEE","#769B63","#CC9900"] + list(matplotlib.colors.cnames.values())
groups = metadata[group_col].unique()
group_map = {g: PALETTE[i] for i,g in enumerate(groups)}
metadata["_color"] = metadata[group_col].map(group_map)

cycle_map = load_rate_cycle_map(cycle_map_path)
print("cycle map (parsed):"); print(cycle_map.to_string(index=False)); print()

rate_block_cycles = {r: find_last_cycle(cycle_map, r) for r in ["0.5C","1C","2C","3C","4C","5C"]}
recovery_cycle = find_recovery_cycle(cycle_map, "5C")
print("Per-rate last cycle:", rate_block_cycles)
print("Recovery cycle:", recovery_cycle); print()

files = [f for f in os.listdir(nda_dir) if f.lower().endswith((".nda",".ndax"))]
print(f"Processing {len(files)} NDA files...")

per_cell_sheets = {}; summary_rows = []
for file in files:
    cell = os.path.splitext(file)[0].split("_")[0]
    if cell not in metadata[bc_col].astype(str).values: cell = os.path.splitext(file)[0]
    if cell not in metadata[bc_col].astype(str).values:
        print(f"  skip (no metadata): {file}"); continue
    el = metadata.loc[metadata[bc_col].astype(str)==cell, group_col].iloc[0]

    rd = NewareNDA.read(os.path.join(nda_dir, file))
    pcs = per_cycle_summary(rd)
    pcs["_rate_label"]  = pcs["Cycle"].map(cycle_map.set_index("_nw_cycle")["_rate"].to_dict())
    pcs["_is_recovery"] = pcs["Cycle"].map(cycle_map.set_index("_nw_cycle")["_is_recov"].to_dict()).fillna(False)
    per_cell_sheets[cell] = pcs

    def _val(c, col):
        if c is None: return np.nan
        s = pcs[pcs["Cycle"]==c]
        if s.empty: return np.nan
        v = pd.to_numeric(s[col], errors="coerce").iloc[0]
        return float(v) if pd.notna(v) else np.nan

    q_05, q_3, q_5 = (_val(rate_block_cycles.get(r), "disch_cap")        for r in ["0.5C","3C","5C"])
    v_05, v_3, v_5 = (_val(rate_block_cycles.get(r), "V_avg_discharge") for r in ["0.5C","3C","5C"])
    q_chg_recov   = _val(recovery_cycle, "chg_cap")
    q_disch_recov = _val(recovery_cycle, "disch_cap")
    if pd.notna(q_chg_recov) and pd.notna(q_disch_recov) and q_disch_recov:
        trapped_Li = 1.0 - q_chg_recov / q_disch_recov
    else:
        trapped_Li = np.nan

    summary_rows.append({
        "Barcode": cell, "Electrolyte": el,
        "rate_cap_ratio_3C_vs_0.5C":          q_3/q_05 if (pd.notna(q_05) and q_05) else np.nan,
        "rate_cap_voltage_drop_3C_vs_0.5C_V": v_3 - v_05 if (pd.notna(v_3) and pd.notna(v_05)) else np.nan,
        "rate_cap_voltage_drop_5C_vs_0.5C_V": v_5 - v_05 if (pd.notna(v_5) and pd.notna(v_05)) else np.nan,
        "rate_cap_trapped_Li_fraction_after_5C": trapped_Li,
        "rate_cap_ratio_5C_vs_0.5C":          q_5/q_05 if (pd.notna(q_05) and q_05) else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
print("=== summary ==="); display(summary_df.head(20))


# 6. Save xlsx (charts FIRST in tab order) + stage1ready.pkl

In [ ]:
import pickle as _pkl
NAME_FONT = {"name": "Helvetica Neue", "size": 14, "bold": True}

def _safe_sheet(name, max_len=31):
    bad = re.compile(r"[\\/*?:\[\]]")
    return bad.sub("_", str(name))[:max_len]

def _build_wide(per_cell, value_col):
    pieces = {bc: df.set_index("Cycle")[value_col] for bc, df in per_cell.items()}
    out = pd.concat(pieces, axis=1); out.index.name = "Cycle"; return out

def _prune_legend(metadata, group_col, bc_col, columns_in_order):
    seen, keep = set(), []
    bc2g = dict(zip(metadata[bc_col].astype(str), metadata[group_col]))
    for i, bc in enumerate(columns_in_order):
        g = bc2g.get(str(bc))
        if g not in seen: seen.add(g); keep.append(i)
    return [i for i in range(len(columns_in_order)) if i not in keep]

metadata_bc_order = metadata[bc_col].astype(str).tolist()
def _reorder(df):
    cols = [c for c in metadata_bc_order if c in df.columns]
    return df[cols]

wide_disch_cap = _reorder(_build_wide(per_cell_sheets, "disch_cap"))
wide_chg_cap   = _reorder(_build_wide(per_cell_sheets, "chg_cap"))
wide_v_disch   = _reorder(_build_wide(per_cell_sheets, "V_avg_discharge"))
wide_ce        = _reorder(_build_wide(per_cell_sheets, "CE_pct"))
wide_cap_ret   = wide_disch_cap.div(wide_disch_cap.iloc[0]) * 100.0
wide_cap_ret.index.name = "Cycle"

# Per-electrolyte avg per C-rate
rate_avg_rows = []
for el, sub in metadata.groupby(group_col):
    cells = sub[bc_col].astype(str).tolist()
    row = {"Electrolyte": el}
    for r in ["0.5C","1C","2C","3C","4C","5C"]:
        cyc = rate_block_cycles.get(r)
        if cyc is None or cyc not in wide_disch_cap.index:
            row[r] = np.nan; continue
        vals = [wide_disch_cap.at[cyc, c] for c in cells if c in wide_disch_cap.columns]
        row[r] = float(np.nanmean(vals)) if vals else np.nan
    rate_avg_rows.append(row)
rate_avg_df = pd.DataFrame(rate_avg_rows)

# Per-electrolyte trapped Li (avg + std for reference, but no error bars in chart)
trapped_avg_rows = []
sdf = summary_df.copy()
for el, sub in sdf.groupby("Electrolyte"):
    vals = pd.to_numeric(sub["rate_cap_trapped_Li_fraction_after_5C"], errors="coerce").dropna()
    trapped_avg_rows.append({
        "Electrolyte": el,
        "trapped_Li_fraction_avg": float(vals.mean()) if len(vals) else np.nan,
        "trapped_Li_fraction_std": float(vals.std()) if len(vals) > 1 else 0.0,
        "n_cells": int(len(vals)),
    })
trapped_avg_df = pd.DataFrame(trapped_avg_rows)
trapped_per_cell = sdf[["Barcode","Electrolyte","rate_cap_trapped_Li_fraction_after_5C"]].copy()
trapped_per_cell.columns = ["Barcode","Electrolyte","trapped_Li_fraction"]

bc_color = dict(zip(metadata[bc_col].astype(str), metadata["_color"]))
delete_idx = _prune_legend(metadata, group_col, bc_col, list(wide_disch_cap.columns))

wide_specs = [
    ("Discharge Capacity Table", wide_disch_cap, "Discharge Capacity (mAh)"),
    ("Charge Capacity Table",    wide_chg_cap,   "Charge Capacity (mAh)"),
    ("Capacity Retention Table", wide_cap_ret,   "Capacity Retention (%)"),
    ("V_avg Discharge Table",    wide_v_disch,   "Average Discharge Voltage (V)"),
    ("CE Table",                 wide_ce,        "Coulombic Efficiency (%)"),
]

with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as writer:
    workbook = writer.book

    # STEP 1: pre-create chartsheets (front tabs)
    chartsheet_handles = {}
    for sname, _, _ in wide_specs:
        chartsheet_handles[sname] = workbook.add_chartsheet(_safe_sheet(sname.replace("Table","Chart")))
    chartsheet_handles["__bar__"]       = workbook.add_chartsheet("Per-Lot Avg Capacity Chart")
    chartsheet_handles["__trap_lot__"]  = workbook.add_chartsheet("Trapped Li (per Lot)")
    chartsheet_handles["__trap_cell__"] = workbook.add_chartsheet("Trapped Li (per Cell)")

    # STEP 2: data sheets
    summary_df.to_excel(writer, sheet_name="summary_features_rate_cap", index=False)
    cycle_map.to_excel(writer, sheet_name="cycle_map_parsed", index=False)
    rate_avg_df.to_excel(writer, sheet_name="per_lot_avg_capacity", index=False)
    trapped_avg_df.to_excel(writer, sheet_name="trapped_Li_per_lot", index=False)
    trapped_per_cell.to_excel(writer, sheet_name="trapped_Li_per_cell", index=False)
    for sname, wdf, _ in wide_specs:
        wdf.to_excel(writer, sheet_name=_safe_sheet(sname))
    for cell, pcs in per_cell_sheets.items():
        pcs.to_excel(writer, sheet_name=_safe_sheet(cell), index=False)

    # STEP 3: build charts
    for sname, wdf, ylabel in wide_specs:
        sheet = _safe_sheet(sname); n_rows = len(wdf)
        chart = workbook.add_chart({"type": "scatter", "subtype": "straight"})
        chart_columns = list(wdf.columns)
        for i, bc in enumerate(chart_columns):
            color = bc_color.get(str(bc), "#808080")
            grp = metadata.loc[metadata[bc_col].astype(str)==str(bc), group_col]
            grp_label = str(grp.iloc[0]) if len(grp) else str(bc)
            chart.add_series({
                "categories": [sheet, 1, 0, n_rows, 0],
                "values":     [sheet, 1, i+1, n_rows, i+1],
                "line":       {"color": color},
                "name":       grp_label,
            })
        x_max = int(wdf.index.max()) if len(wdf) else 1
        all_vals = pd.to_numeric(wdf.stack(), errors="coerce").dropna()
        if len(all_vals) > 0:
            y_min, y_max = float(all_vals.min()), float(all_vals.max())
            pad = max(0.05*(y_max - y_min), 1e-6); y_min -= pad; y_max += pad
        else:
            y_min, y_max = 0, 1
        chart.set_x_axis({"name": "Cycle Number", "name_font": NAME_FONT, "min": 1, "max": x_max})
        chart.set_y_axis({"name": ylabel, "name_font": NAME_FONT, "min": y_min, "max": y_max})
        chart.set_legend({"delete_series": delete_idx})
        chartsheet_handles[sname].set_chart(chart)

    # Per-Lot avg capacity bar chart
    rate_cols = ["0.5C","1C","2C","3C","4C","5C"]
    if all(c in rate_avg_df.columns for c in rate_cols):
        bar = workbook.add_chart({"type":"column"}); n_lots = len(rate_avg_df)
        for j, r in enumerate(rate_cols):
            bar.add_series({
                "categories": ["per_lot_avg_capacity", 1, 0, n_lots, 0],
                "values":     ["per_lot_avg_capacity", 1, 1+j, n_lots, 1+j],
                "name":       r,
            })
        bar.set_title({"name": "Avg Discharge Capacity per C-rate (per electrolyte)",
                       "name_font": {"name":"Helvetica Neue","size":13,"bold":True}})
        bar.set_x_axis({"name": "Electrolyte", "name_font": NAME_FONT})
        bar.set_y_axis({"name": "Discharge Capacity (mAh)", "name_font": NAME_FONT})
        chartsheet_handles["__bar__"].set_chart(bar)

    # Trapped Li per-lot - clean avg bar (no error bars; std is in data sheet)
    n_lots_t = len(trapped_avg_df)
    if n_lots_t > 0:
        bar2 = workbook.add_chart({"type":"column"})
        bar2.add_series({
            "categories": ["trapped_Li_per_lot", 1, 0, n_lots_t, 0],
            "values":     ["trapped_Li_per_lot", 1, 1, n_lots_t, 1],
            "name":       "Trapped Li fraction (lot avg)",
            "fill":       {"color": "#1f77b4"},
        })
        bar2.set_title({"name": "Trapped Li fraction after 5C  (per electrolyte avg)",
                        "name_font": {"name":"Helvetica Neue","size":13,"bold":True}})
        bar2.set_x_axis({"name": "Electrolyte", "name_font": NAME_FONT})
        bar2.set_y_axis({"name": "1 - Q_chg / Q_disch", "name_font": NAME_FONT, "min": 0})
        bar2.set_legend({"none": True})
        chartsheet_handles["__trap_lot__"].set_chart(bar2)

    # Trapped Li per-cell bar
    n_cells_t = len(trapped_per_cell)
    if n_cells_t > 0:
        bar3 = workbook.add_chart({"type":"column"})
        bar3.add_series({
            "categories": ["trapped_Li_per_cell", 1, 0, n_cells_t, 0],
            "values":     ["trapped_Li_per_cell", 1, 2, n_cells_t, 2],
            "name":       "Trapped Li fraction",
            "points":     [{"fill": {"color": bc_color.get(str(bc), "#808080")}}
                           for bc in trapped_per_cell["Barcode"]],
        })
        bar3.set_title({"name": "Trapped Li fraction after 5C  (per cell, color = electrolyte)",
                        "name_font": {"name":"Helvetica Neue","size":13,"bold":True}})
        bar3.set_x_axis({"name": "Barcode", "name_font": NAME_FONT, "num_font": {"rotation": -45}})
        bar3.set_y_axis({"name": "1 - Q_chg / Q_disch", "name_font": NAME_FONT, "min": 0})
        bar3.set_legend({"none": True})
        chartsheet_handles["__trap_cell__"].set_chart(bar3)

    chartsheet_handles[wide_specs[0][0]].activate()
    chartsheet_handles[wide_specs[0][0]].set_first_sheet()

print("xlsx saved:", out_xlsx)

with open(out_pkl, "wb") as f:
    _pkl.dump({"rate_cap_summary": summary_df}, f)
print("stage1ready.pkl saved:", out_pkl)
